# Setup and shared functions

Constants, input data and functions used by notebooks 01-05, each of which runs this notebook first with `%run 00_setup_and_functions.ipynb`. Running this notebook on its own only loads and checks the data.

## Configuration
Paths, experimental constants, colours and figure style shared by all notebooks. The notebooks are run from the `notebooks/` folder; inputs are read from `data/` and outputs written to `results/`.

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
ROOT = Path.cwd().resolve().parent          # notebooks are run from notebooks/
DATA_DIR = ROOT / "data"
FIGURE_DIR = ROOT / "results" / "figures"
TABLE_DIR = ROOT / "results" / "tables"          # Supplementary Tables S1-S14
STATS_DIR = ROOT / "results" / "statistics"      # further statistics reported in the text
for _p in (FIGURE_DIR, TABLE_DIR, STATS_DIR):
    _p.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Experimental design
# ---------------------------------------------------------------------------
# Community codes: three- or 16-species community containing ancestral (ANC) or
# streptomycin-resistance-evolved (EVO) Aeromonas caviae HAMBI 1972.
COMMUNITY_TYPES = ["ANC_THREE", "EVO_THREE", "ANC_SIXTEEN", "EVO_SIXTEEN"]
RICHNESS_PAIRS = [("3", "ANC_THREE", "EVO_THREE"), ("16", "ANC_SIXTEEN", "EVO_SIXTEEN")]
FOCAL_SPECIES = "HAMBI_1972"
CORE_THREE = ["HAMBI_1287", "HAMBI_1972", "HAMBI_1977"]
LOW_DOSE = 2.0

RANDOM_SEED = 20260915

# Community-state segmentation: boundary penalty (lambda) and maximum Bray-Curtis
# diameter of a state; the sensitivity analysis varies both (Figure S1).
BASE_BOUNDARY_PENALTY = 0.15
BASE_MAX_BC = 0.40
SENSITIVITY_SETTINGS = [(0.05, 0.40), (0.15, 0.40), (0.25, 0.40), (0.15, 0.10), (0.15, 0.70)]

# Log-ratio analyses: zero replacement constant, and the detection threshold for the
# species-level CLR basis (taxa detected in more than half of 16-species samples).
CLR_PSEUDOCOUNT = 1e-6
SPECIES_LEVEL_MIN_PREVALENCE = 0.5

# Streptomycin MIC classes (ug/ml); ">160" is kept as a censored class.
MIC_SCORE = {"10": 0, "20": 1, "40": 2, "160": 3, ">160": 4}
MIC_LABEL_ORDER = ["10", "20", "40", "160", ">160"]
HIGH_MIC_MIN_SCORE = MIC_SCORE["160"]

# ---------------------------------------------------------------------------
# Labels
# ---------------------------------------------------------------------------
CARBON_AXIS_LABEL = "Glucose-derived carbon (g C L$^{-1}$)"
STREP_AXIS_LABEL = "Streptomycin (µg ml$^{-1}$)"
TREATMENT_LABELS = {
    "ANC_THREE": "3-species community containing ancestral A. caviae",
    "EVO_THREE": "3-species community containing resistance-evolved A. caviae",
    "ANC_SIXTEEN": "16-species community containing ancestral A. caviae",
    "EVO_SIXTEEN": "16-species community containing resistance-evolved A. caviae",
}
RICHNESS_LABELS = {"3": "3-species community", "16": "16-species community"}
PANEL_TREATMENT_LABELS = {
    "ANC_THREE": "3-species community\ncontaining ancestral\n$\\it{Aeromonas\\ caviae}$",
    "EVO_THREE": "3-species community\ncontaining resistance-evolved\n$\\it{Aeromonas\\ caviae}$",
    "ANC_SIXTEEN": "16-species community\ncontaining ancestral\n$\\it{Aeromonas\\ caviae}$",
    "EVO_SIXTEEN": "16-species community\ncontaining resistance-evolved\n$\\it{Aeromonas\\ caviae}$",
}

# ---------------------------------------------------------------------------
# Colours
# ---------------------------------------------------------------------------
# History colours (orange = ancestral, purple = resistance-evolved) differ in lightness as
# well as hue and remain distinct under common forms of colour-vision deficiency. Every
# diverging map of an evolved-minus-ancestral difference uses the same two colours as its
# poles (ColorBrewer PuOr), including Shannon Index differences.
ANCESTRAL_COLOR = "#B35806"
EVOLVED_COLOR = "#542788"
CMAP_HISTORY = LinearSegmentedColormap.from_list(
    "history_diverging", ["#B35806", "#F1A340", "#F7F7F7", "#998EC3", "#542788"])
CMAP_SEQ = "cividis"
TRAIT_RATE_COLOR = "#2F6F62"
TRAIT_LAG_COLOR = "#9CC3B8"
REFERENCE_GREY = "#C7C7C7"
SPECIES_COLORS = {
    "HAMBI_1972": "#D1495B", "HAMBI_1287": "#6A4C93", "HAMBI_1977": "#277DA1",
    "HAMBI_0006": "#43AA8B", "HAMBI_0105": "#F9C74F", "HAMBI_0403": "#90BE6D",
    "HAMBI_1292": "#F9844A", "HAMBI_1299": "#B5179E", "HAMBI_1842": "#4D908E",
    "HAMBI_1896": "#8D6E63", "HAMBI_2160": "#577590", "HAMBI_2164": "#F15BB5",
    "HAMBI_2443": "#7209B7", "HAMBI_2659": "#00A6D6", "HAMBI_3031": "#BDB2FF",
    "HAMBI_3237": "#4361EE",
}

# ---------------------------------------------------------------------------
# Figure style
# ---------------------------------------------------------------------------
MM = 1 / 25.4
plt.ioff()
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 7.5,
    "axes.titlesize": 8.5,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6.5,
    "figure.titlesize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.bbox": "tight",
})

## Input data
Reads `data/community_composition.tsv` and `data/species.tsv`, checks that the design is complete, and defines the objects used throughout the notebooks:

* `data_extended`: long table, one row per sample x species (relative abundance)
* `sample_compositions`: one row per sample, with species columns, Shannon Index and focal-species abundance
* `SPECIES`, `CARBON_VALUES`, `STREPTOMYCIN_VALUES`, `COMPLETE_REPLICATES` and the grid sizes

In [2]:
import numpy as np
import pandas as pd


COMPOSITION_FILE = DATA_DIR / "community_composition.tsv"
SPECIES_FILE = DATA_DIR / "species.tsv"
GROWTH_FILE = DATA_DIR / "monoculture_growth.tsv"

# ---------------------------------------------------------------------------
# Species
# ---------------------------------------------------------------------------
species_info = pd.read_csv(SPECIES_FILE, sep="\t", dtype={"streptomycin_MIC_ancestral": str,
                                                          "streptomycin_MIC_resistance_evolved": str})
SPECIES_NAMES = {r.species: f"{r.species_name} {r.strain}" for r in species_info.itertuples()}
SPECIES_SHORT = dict(zip(species_info.species, species_info.species_name))
MIC_ANCESTRAL = dict(zip(species_info.species, species_info.streptomycin_MIC_ancestral))
MIC_EVOLVED = dict(zip(species_info.species, species_info.streptomycin_MIC_resistance_evolved))

# ---------------------------------------------------------------------------
# Endpoint community composition
# ---------------------------------------------------------------------------
_raw = pd.read_csv(COMPOSITION_FILE, sep="\t")
_meta_columns = ["community", "replicate", "carbon_level", "streptomycin", "carbon_g_L"]
_raw = _raw.loc[_raw["community"].isin(COMMUNITY_TYPES)].copy()

SPECIES = sorted(_raw["species"].unique())
if len(SPECIES) != 16 or set(SPECIES) != set(SPECIES_NAMES):
    raise ValueError("Species in the composition table do not match data/species.tsv")
if (_raw.groupby("sample")[_meta_columns].nunique() > 1).any().any():
    raise ValueError("Inconsistent metadata within a sample")
if _raw.duplicated(["sample", "species"]).any():
    raise ValueError("Duplicate sample x species rows")

_sample_meta = _raw[["sample"] + _meta_columns].drop_duplicates("sample").set_index("sample")
_full_index = pd.MultiIndex.from_product([_sample_meta.index, SPECIES], names=["sample", "species"])
data_extended = _raw.set_index(["sample", "species"]).reindex(_full_index)
data_extended["relative_abundance"] = data_extended["relative_abundance"].fillna(0.0)
_sample_index = data_extended.index.get_level_values("sample")
for _c in _meta_columns:
    data_extended[_c] = _sample_index.map(_sample_meta[_c])
data_extended = data_extended.reset_index()[["sample", "species", "relative_abundance"] + _meta_columns]

if not np.allclose(data_extended.groupby("sample")["relative_abundance"].sum(), 1.0, atol=1e-8):
    raise ValueError("Sample compositions do not sum to one")

CARBON_VALUES = np.array(sorted(data_extended["carbon_g_L"].unique()), dtype=float)
STREPTOMYCIN_VALUES = np.array(sorted(data_extended["streptomycin"].unique()), dtype=float)
N_CARBON, N_STREP = len(CARBON_VALUES), len(STREPTOMYCIN_VALUES)
EXPECTED_N_CELLS = N_CARBON * N_STREP

_coverage = (data_extended[["sample", "community", "replicate"]].drop_duplicates()
             .groupby(["community", "replicate"]).size())
if not (_coverage == EXPECTED_N_CELLS).all():
    raise ValueError("Every replicate landscape must cover the full 16 x 4 grid")
COMPLETE_REPLICATES = {c: sorted(_coverage.loc[c].index.astype(int)) for c in COMMUNITY_TYPES}

# One row per sample with the 16 relative abundances and derived quantities.
sample_compositions = data_extended.pivot_table(
    index=["sample", "community", "replicate", "carbon_g_L", "streptomycin"],
    columns="species", values="relative_abundance", fill_value=0.0,
).reset_index()


def _shannon(p):
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())


sample_compositions["shannon"] = [_shannon(p) for p in sample_compositions[SPECIES].to_numpy(float)]
sample_compositions["p1972"] = sample_compositions[FOCAL_SPECIES]


def composition_grid(community, replicate=None):
    """Composition array (carbon x streptomycin x species) for one replicate landscape,
    or the mean landscape across replicates when `replicate` is None."""
    df = data_extended.loc[data_extended.community == community]
    if replicate is not None:
        df = df.loc[df.replicate == replicate]
    index = pd.MultiIndex.from_product([CARBON_VALUES, STREPTOMYCIN_VALUES, SPECIES],
                                       names=["carbon_g_L", "streptomycin", "species"])
    grid = df.groupby(["carbon_g_L", "streptomycin", "species"])["relative_abundance"].mean().reindex(index)
    if grid.isna().any():
        raise ValueError(f"Incomplete grid for {community}, replicate {replicate}")
    return grid.to_numpy(float).reshape(N_CARBON, N_STREP, len(SPECIES))


# ---------------------------------------------------------------------------
# Monoculture growth traits
# ---------------------------------------------------------------------------
GROWTH_TRAITS = ["max_growth_rate", "yield", "auc", "lag_time_h"]
GROWTH_STRAIN_LABELS = {
    "HAMBI_1287": "Citrobacter koseri",
    "HAMBI_1972_ANC": "Aeromonas caviae\nancestral",
    "HAMBI_1972_EVO": "Aeromonas caviae\nresistance-evolved",
    "HAMBI_1977": "Pseudomonas chlororaphis",
}
# Strain of each core species present in the three-species community of each history.
GROWTH_STRAIN_BY_COMMUNITY = {
    "ANC_THREE": {"HAMBI_1287": "HAMBI_1287", "HAMBI_1972": "HAMBI_1972_ANC", "HAMBI_1977": "HAMBI_1977"},
    "EVO_THREE": {"HAMBI_1287": "HAMBI_1287", "HAMBI_1972": "HAMBI_1972_EVO", "HAMBI_1977": "HAMBI_1977"},
}


def load_growth():
    """Monoculture growth traits: one row per strain x carbon x streptomycin x replicate."""
    growth = pd.read_csv(GROWTH_FILE, sep="\t")
    if growth.duplicated(["strain", "replicate", "carbon_level", "streptomycin"]).any():
        raise ValueError("Expected one row per strain x replicate x carbon x streptomycin")
    if not (growth.groupby(["strain", "carbon_level", "streptomycin"]).size() == 6).all():
        raise ValueError("Expected six monoculture replicates per strain and condition")
    return growth

## Statistical tests
Exact randomization tests, bootstrap intervals and multiplicity corrections.

The inferential unit is the constructed landscape (one biological replicate community measured across all 64 environments). With eight landscapes per history, two-group tests enumerate all C(16, 8) = 12,870 allocations of history labels; one-sample tests enumerate all 2^8 sign flips.

In [3]:
from functools import lru_cache
from itertools import combinations, product

import numpy as np
from scipy.stats import rankdata
from statsmodels.stats.multitest import multipletests



@lru_cache(maxsize=None)
def independent_masks(n_total, n_x):
    """All allocations of n_x of n_total units to the first group (rows of a 0/1 matrix)."""
    combs = list(combinations(range(n_total), n_x))
    mask = np.zeros((len(combs), n_total), dtype=float)
    for i, idx in enumerate(combs):
        mask[i, list(idx)] = 1.0
    return mask


@lru_cache(maxsize=None)
def sign_matrix(n):
    return np.asarray(list(product([-1.0, 1.0], repeat=n)), dtype=float)


def exact_mean_difference_test(x, y, paired=False):
    """Difference in means (y - x) with a two-sided exact randomization P value."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]; y = y[np.isfinite(y)]
    if paired:
        d = y - x
        null = np.abs((sign_matrix(len(d)) * d[None, :]).mean(axis=1))
        return float(d.mean()), float(np.mean(null >= abs(d.mean()) - 1e-15))
    pooled = np.r_[x, y]
    nx, ny = len(x), len(y)
    sum_x = independent_masks(len(pooled), nx) @ pooled
    null = np.abs((pooled.sum() - sum_x) / ny - sum_x / nx)
    obs = abs(y.mean() - x.mean())
    return float(y.mean() - x.mean()), float(np.mean(null >= obs - 1e-15))


def exact_centroid_distance_test(X, Y):
    """Euclidean distance between group centroids with an exact allocation P value."""
    X = np.asarray(X, float); Y = np.asarray(Y, float)
    pooled = np.vstack([X, Y]); nx, ny = len(X), len(Y)
    obs = float(np.linalg.norm(Y.mean(axis=0) - X.mean(axis=0)))
    sum_x = independent_masks(nx + ny, nx) @ pooled
    null = np.linalg.norm((pooled.sum(axis=0)[None, :] - sum_x) / ny - sum_x / nx, axis=1)
    return obs, float(np.mean(null >= obs - 1e-15))


def exact_signflip(differences):
    """Mean of paired differences (or slopes) with an exact sign-flip P value."""
    d = np.asarray(differences, float)
    d = d[np.isfinite(d)]
    null = np.abs((sign_matrix(len(d)) * d[None, :]).mean(axis=1))
    return float(d.mean()), float(np.mean(null >= abs(d.mean()) - 1e-15))


def bootstrap_diff_ci(x, y, paired=False, n_boot=10000, seed=RANDOM_SEED):
    """Percentile 95% CI of mean(y) - mean(x), resampling landscapes within each group."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    rng = np.random.default_rng(seed)
    if paired:
        d = y - x
        vals = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(axis=1)
    else:
        ix = rng.integers(0, len(x), size=(n_boot, len(x)))
        iy = rng.integers(0, len(y), size=(n_boot, len(y)))
        vals = y[iy].mean(axis=1) - x[ix].mean(axis=1)
    return np.quantile(vals, [0.025, 0.975]).astype(float)


def bootstrap_mean_ci(values, n_boot=10000, seed=RANDOM_SEED):
    values = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(values), size=(n_boot, len(values)))
    return np.quantile(values[idx].mean(axis=1), [0.025, 0.975]).astype(float)


def holm(pvalues):
    return multipletests(np.asarray(pvalues, float), method="holm")[1]


def bh(pvalues):
    return multipletests(np.asarray(pvalues, float), method="fdr_bh")[1]


def permanova_two_group(X, labels):
    """Pseudo-F and R2 of a two-group PERMANOVA in Euclidean space."""
    X = np.asarray(X, float); labels = np.asarray(labels, int)
    grand = X.mean(axis=0); ss_total = float(((X - grand) ** 2).sum())
    ss_between = 0.0
    for g in np.unique(labels):
        xg = X[labels == g]
        ss_between += len(xg) * float(((xg.mean(axis=0) - grand) ** 2).sum())
    ss_within = ss_total - ss_between
    df_between = len(np.unique(labels)) - 1; df_within = len(X) - len(np.unique(labels))
    return (ss_between / df_between) / (ss_within / df_within), ss_between / ss_total


def exact_permanova(X, labels):
    """Exact two-group PERMANOVA over all allocations of the group labels."""
    labels = np.asarray(labels, int); obs_F, R2 = permanova_two_group(X, labels)
    n1 = int(labels.sum()); n = len(labels); exceed = 0; total = 0
    for idx in combinations(range(n), n1):
        lab = np.zeros(n, int); lab[list(idx)] = 1
        F, _ = permanova_two_group(X, lab); exceed += (F >= obs_F - 1e-12); total += 1
    return obs_F, R2, exceed / total, total


def exact_permanova_distance(D, n_first=8):
    """Exact two-group PERMANOVA from a distance matrix (Anderson 2001). Rows of D are
    ordered with the n_first units of the first group (ancestral history) first."""
    D2 = np.asarray(D, float) ** 2; n = len(D2); n_second = n - n_first
    alloc = independent_masks(n, n_first)
    sst = D2[np.triu_indices(n, 1)].sum() / n
    ssw_all = (np.einsum("ki,ij,kj->k", alloc, D2, alloc) / 2 / n_first
               + np.einsum("ki,ij,kj->k", 1 - alloc, D2, 1 - alloc) / 2 / n_second)
    F_all = (sst - ssw_all) / (ssw_all / (n - 2))
    first = np.r_[np.ones(n_first), np.zeros(n_second)]
    ssw = (first @ D2 @ first) / 2 / n_first + ((1 - first) @ D2 @ (1 - first)) / 2 / n_second
    F = (sst - ssw) / (ssw / (n - 2))
    return float(F), float((sst - ssw) / sst), float((F_all >= F - 1e-12).mean())


def partial_spearman(x, y, z):
    """Spearman correlation of x and y controlling for z (Pearson correlation of rank residuals)."""
    rx, ry, rz = (rankdata(v) for v in (x, y, z))

    def resid(a, b):
        B = np.c_[np.ones_like(b), b]
        return a - B @ np.linalg.lstsq(B, a, rcond=None)[0]
    return float(np.corrcoef(resid(rx, rz), resid(ry, rz))[0, 1])


def mic_permutation_p(scores, responses, n_perm=200_000, seed=RANDOM_SEED):
    """Spearman correlation between MIC class and response, with a Monte Carlo P value
    from permutations of the MIC labels (ties receive average ranks)."""
    sr = rankdata(np.asarray(scores, float)).astype(float)
    rr = rankdata(np.asarray(responses, float)).astype(float)
    sr0 = sr - sr.mean(); rr0 = rr - rr.mean()
    denom = np.sqrt(np.sum(sr0 ** 2) * np.sum(rr0 ** 2))
    rho = float(np.dot(sr0, rr0) / denom)
    rng = np.random.default_rng(seed)
    count = 0
    for start in range(0, n_perm, 100_000):
        m = min(100_000, n_perm - start)
        perms = sr[np.argsort(rng.random((m, len(sr))), axis=1)]
        rhos = ((perms - perms.mean(axis=1, keepdims=True)) @ rr0) / denom
        count += int(np.sum(np.abs(rhos) >= abs(rho) - 1e-15))
    return rho, float((count + 1) / (n_perm + 1))

## Community-state segmentation
Entropy-penalized spatial segmentation of community-composition landscapes, and descriptors of the resulting state boundaries.

**Method by Dovydas Kičiatovas and Mikko Kivikoski.** Generative AI tools were used to assist in developing the algorithm and writing the code.

Each landscape is a 16 carbon x 4 streptomycin grid of community compositions $p_i$ (relative abundances summing to one). Grid cells are nodes of a four-neighbour graph: cells that differ by one step along one axis are adjacent; diagonals are not.

A partition of the grid into spatially connected states is scored by

$$J = \sum_{C} L(C) + \lambda B,$$

* $L(C) = \sum_{i \in C} \mathrm{KL}(p_i \,\|\, m_C)$ is a generalized Jensen-Shannon loss, where $m_C$ is the mean composition of state $C$. It is zero when all communities in the state are identical and grows as they diverge. (Shannon entropy of single communities would instead favour species-poor communities, not homogeneous states.)
* $B$ is the number of adjacent cell pairs assigned to different states, so $\lambda$ sets the cost of a boundary and hence the resolution of the segmentation.

Optimization is greedy region merging. Every cell starts as its own state ($J = \lambda B$). At each step, for every pair of adjacent states $A$, $B$ sharing $b(A, B)$ grid edges, the change in the objective of merging them is

$$\Delta J = L(A \cup B) - L(A) - L(B) - \lambda\, b(A, B),$$

and the pair with the most negative $\Delta J$ is merged. Because only adjacent states are merged, every state stays spatially connected. A merge is rejected if the merged state would have a Bray-Curtis diameter (largest pairwise Bray-Curtis distance between its cells) above the threshold; this prevents chains of gradually changing communities from forming one state whose ends differ strongly. Merging stops when no admissible merge lowers $J$, so the number of states is not fixed in advance but follows from $\lambda$, the diameter threshold and the data.

In [4]:
from math import comb

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist


def _validate(data):
    data = np.asarray(data, dtype=float)
    if data.ndim != 3:
        raise ValueError("data must have shape (carbon, streptomycin, species)")
    if not np.all(np.isfinite(data)) or np.any(data < 0):
        raise ValueError("Invalid composition values")
    totals = data.sum(axis=2, keepdims=True)
    if np.any(totals <= 0):
        raise ValueError("Every grid cell must have positive total abundance")
    return data / totals


def generalized_js_loss(compositions):
    """L(C): summed KL divergence of each composition from the state's mean composition."""
    if len(compositions) <= 1:
        return 0.0
    centroid = compositions.mean(axis=0); centroid /= centroid.sum()
    return float(np.sum(np.sum(compositions * (np.log(compositions) - np.log(centroid)), axis=1)))


def bray_curtis_diameter(compositions):
    return 0.0 if len(compositions) <= 1 else float(pdist(compositions, metric="braycurtis").max())


def make_grid_edges(n_carbon, n_strep):
    """Four-neighbour edges between cells indexed g * n_strep + s."""
    edges = []
    for g in range(n_carbon):
        for s in range(n_strep):
            if g + 1 < n_carbon:
                edges.append((g * n_strep + s, (g + 1) * n_strep + s))
            if s + 1 < n_strep:
                edges.append((g * n_strep + s, g * n_strep + s + 1))
    return edges


def _adjacent_region_pairs(cell_to_region, edges):
    shared = {}
    for a, b in edges:
        ra, rb = cell_to_region[a], cell_to_region[b]
        if ra != rb:
            pair = tuple(sorted((ra, rb)))
            shared[pair] = shared.get(pair, 0) + 1
    return shared


def entropy_penalized_spatial_segmentation(data, boundary_penalty=0.15, max_bray_curtis_diameter=0.40,
                                           pseudocount=1e-12):
    """Segment a (carbon x streptomycin x species) composition array into community states.

    Returns a dict with the state label of every cell (`labels_grid`), the number of states,
    state sizes and mean compositions, and the number of boundary edges.
    """
    data = _validate(data)
    ng, ns, nsp = data.shape
    nc = ng * ns
    original = data.reshape(nc, nsp)
    # A tiny pseudocount keeps the logarithms in L(C) finite for absent species.
    positive = original + pseudocount
    positive = positive / positive.sum(axis=1, keepdims=True)
    edges = make_grid_edges(ng, ns)
    regions = {i: {i} for i in range(nc)}
    losses = {i: 0.0 for i in range(nc)}
    cell_to_region = np.arange(nc, dtype=int)
    next_id = nc
    while True:
        best = None
        best_change = 0.0
        for (ra, rb), n_shared in _adjacent_region_pairs(cell_to_region, edges).items():
            members = regions[ra] | regions[rb]
            idx = np.array(sorted(members), int)
            merged_loss = generalized_js_loss(positive[idx])
            change = merged_loss - losses[ra] - losses[rb] - boundary_penalty * n_shared
            if max_bray_curtis_diameter is not None and bray_curtis_diameter(original[idx]) > max_bray_curtis_diameter:
                continue
            if change < best_change:
                best_change = change
                best = (ra, rb, members, merged_loss)
        if best is None:
            break
        ra, rb, members, merged_loss = best
        del regions[ra], regions[rb], losses[ra], losses[rb]
        regions[next_id] = members
        losses[next_id] = merged_loss
        for c in members:
            cell_to_region[c] = next_id
        next_id += 1

    labels = np.empty(nc, dtype=int)
    for new_label, region_id in enumerate(sorted(regions)):
        labels[list(regions[region_id])] = new_label
    sizes, centroids = {}, {}
    for cid in np.unique(labels):
        idx = np.flatnonzero(labels == cid)
        sizes[int(cid)] = int(len(idx))
        centroids[int(cid)] = original[idx].mean(axis=0)
    return {
        "labels_grid": labels.reshape(ng, ns), "n_clusters": len(sizes),
        "cluster_sizes": sizes, "cluster_centroids": centroids,
        "total_boundary_edges": int(sum(labels[a] != labels[b] for a, b in edges)),
    }


# ---------------------------------------------------------------------------
# Boundary descriptors
# ---------------------------------------------------------------------------
def boundary_vector(labels):
    """0/1 vector over the 108 grid edges (1 = the edge separates two states)."""
    L = np.asarray(labels)
    ng, ns = L.shape
    v = []
    for i in range(ng):
        for j in range(ns):
            if i + 1 < ng:
                v.append(L[i, j] != L[i + 1, j])
            if j + 1 < ns:
                v.append(L[i, j] != L[i, j + 1])
    return np.asarray(v, float)


def boundary_incidence(labels):
    """Fraction of each cell's adjacent edges that cross a state boundary."""
    labels = np.asarray(labels)
    ng, ns = labels.shape
    hit = np.zeros((ng, ns)); possible = np.zeros((ng, ns))
    for g in range(ng):
        for s in range(ns):
            for dg, ds in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                a, b = g + dg, s + ds
                if 0 <= a < ng and 0 <= b < ns:
                    possible[g, s] += 1
                    hit[g, s] += labels[g, s] != labels[a, b]
    return hit / possible


def adjusted_rand(a, b):
    """Adjusted Rand index between two partitions of the same cells."""
    a = np.ravel(a); b = np.ravel(b); n = len(a)
    ct = pd.crosstab(a, b).to_numpy()
    sij = sum(comb(int(x), 2) for x in ct.ravel())
    sa = sum(comb(int(x), 2) for x in ct.sum(1)); sb = sum(comb(int(x), 2) for x in ct.sum(0))
    expected = sa * sb / comb(n, 2); maximum = (sa + sb) / 2
    return 1.0 if maximum == expected else (sij - expected) / (maximum - expected)


def dominant_species_map(result):
    """Index of the most abundant species in the mean composition of each cell's state."""
    labels = np.asarray(result["labels_grid"])
    out = np.empty_like(labels, dtype=int)
    for cid in np.unique(labels):
        out[labels == cid] = int(np.argmax(np.asarray(result["cluster_centroids"][int(cid)], float)))
    return out

## Compositional responses and resistance filtering
Antibiotic responses in log-ratio (CLR) space and resistance filtering. A response is always measured against the matched no-antibiotic sample of the same replicate landscape and carbon level, which removes the carbon-specific baseline.

In [5]:
import numpy as np
import pandas as pd


KEYS = ["community", "replicate", "carbon_g_L"]
NONFOCAL_SPECIES = [s for s in SPECIES if s != FOCAL_SPECIES]
NONZERO_DOSES = [2.0, 20.0, 200.0]

# Species-level CLR basis: taxa detected in more than half of the 16-species samples.
# With rarely detected taxa in the basis, the CLR geometric mean is governed by how many
# of them sit at the zero-replacement constant rather than by composition.
_sixteen = sample_compositions.loc[sample_compositions.community.isin(["ANC_SIXTEEN", "EVO_SIXTEEN"])]
SPECIES_PREVALENCE = {s: float((_sixteen[s] > 0).mean()) for s in SPECIES}
CLR_SPECIES = [s for s in SPECIES if SPECIES_PREVALENCE[s] > SPECIES_LEVEL_MIN_PREVALENCE]


def clr(P, pseudocount=CLR_PSEUDOCOUNT):
    """Centred log-ratio transform of the rows of P after zero replacement and closure."""
    P = np.asarray(P, float) + pseudocount
    P = P / P.sum(axis=1, keepdims=True)
    logP = np.log(P)
    return logP - logP.mean(axis=1, keepdims=True)


def clr_response(species, prefix):
    """CLR coordinates of `species` (closed among themselves) and their change from 0 ug/ml
    at each non-zero streptomycin dose. Returns (sample-level responses, response columns)."""
    cols = [f"{prefix}_{s}" for s in species]
    df = sample_compositions[["sample"] + KEYS + ["streptomycin"]].copy()
    df[cols] = clr(sample_compositions[species].to_numpy(float))
    base = df.loc[df.streptomycin == 0, KEYS + cols].rename(columns={c: f"base_{c}" for c in cols})
    stressed = df.loc[df.streptomycin > 0].merge(base, on=KEYS, how="left", validate="many_to_one")
    dcols = [f"d_{c}" for c in cols]
    for c, dc in zip(cols, dcols):
        stressed[dc] = stressed[c] - stressed[f"base_{c}"]
    return stressed, dcols


def whole_community_clr_response():
    """Carbon-averaged CLR response of each replicate landscape (species-level CLR basis)."""
    stressed, dcols = clr_response(CLR_SPECIES, "clr")
    rep = stressed.groupby(["community", "replicate", "streptomycin"])[dcols].mean().reset_index()
    return rep, dcols


def species_clr_responses(dclr_rep):
    """Mean CLR response of each species per community and dose, with its MIC class."""
    rows = []
    for community in COMMUNITY_TYPES:
        mic = MIC_ANCESTRAL if community.startswith("ANC") else MIC_EVOLVED
        for dose in NONZERO_DOSES:
            sub = dclr_rep.loc[(dclr_rep.community == community) & (dclr_rep.streptomycin == dose)]
            for s in CLR_SPECIES:
                vals = sub[f"d_clr_{s}"].to_numpy(float)
                rows.append({"community": community, "streptomycin": dose, "species": s,
                             "species_name": SPECIES_NAMES[s], "mean_delta_clr": vals.mean(),
                             "MIC_class": mic[s], "MIC_score": MIC_SCORE[mic[s]]})
    return pd.DataFrame(rows)


def species_clr_history_contrasts(dclr_rep):
    """Evolved-minus-ancestral CLR response of each species in the 16-species community,
    Benjamini-Hochberg corrected across the species of the CLR basis at each dose."""
    rows = []
    for dose in NONZERO_DOSES:
        family = []
        for s in CLR_SPECIES:
            x = dclr_rep.loc[(dclr_rep.community == "ANC_SIXTEEN") & (dclr_rep.streptomycin == dose), f"d_clr_{s}"].to_numpy(float)
            y = dclr_rep.loc[(dclr_rep.community == "EVO_SIXTEEN") & (dclr_rep.streptomycin == dose), f"d_clr_{s}"].to_numpy(float)
            diff, p = exact_mean_difference_test(x, y)
            lo, hi = bootstrap_diff_ci(x, y, seed=RANDOM_SEED + 8000 + int(dose) + SPECIES.index(s))
            family.append({"streptomycin": dose, "species": s, "species_name": SPECIES_NAMES[s],
                           "evolved_minus_ancestral_delta_clr": diff, "ci95_low": lo, "ci95_high": hi,
                           "p_exact": p})
        for r, q in zip(family, bh([r["p_exact"] for r in family])):
            r["q_bh_across_species"] = float(q)
        rows += family
    return pd.DataFrame(rows)


def resistance_filtering():
    """Share of non-focal abundance carried by high-MIC taxa (MIC >= 160 ug/ml) and the
    non-focal community-weighted MIC-class score, per replicate landscape and dose
    (averaged across carbon). A. caviae is excluded so that the measure describes the
    rest of the community. Returns (landscape values, within-history dose tests,
    between-history tests)."""
    metrics = ["community_weighted_MIC_score_excl_focal", "high_MIC_share_excl_focal"]
    landscape_rows, enrichment_rows = [], []
    for community in ["ANC_SIXTEEN", "EVO_SIXTEEN"]:
        mic = MIC_ANCESTRAL if community.startswith("ANC") else MIC_EVOLVED
        sub = sample_compositions.loc[sample_compositions.community == community].copy()
        denom = sub[NONFOCAL_SPECIES].sum(axis=1)
        sub["community_weighted_MIC_score_excl_focal"] = sum(sub[s] * MIC_SCORE[mic[s]] for s in NONFOCAL_SPECIES) / denom
        sub["high_MIC_share_excl_focal"] = sum(sub[s] * (MIC_SCORE[mic[s]] >= HIGH_MIC_MIN_SCORE) for s in NONFOCAL_SPECIES) / denom
        rep = sub.groupby(["replicate", "streptomycin"])[metrics].mean().reset_index()
        rep["community"] = community
        landscape_rows.append(rep)
        # Each non-zero dose against the no-antibiotic value of the same landscape.
        for metric in metrics:
            pvt = rep.pivot(index="replicate", columns="streptomycin", values=metric)
            family = []
            for dose in NONZERO_DOSES:
                d = (pvt[dose] - pvt[0.0]).dropna().to_numpy(float)
                effect, p = exact_signflip(d)
                family.append({"community": community, "metric": metric, "streptomycin": dose,
                               "no_antibiotic_mean": float(pvt[0.0].mean()), "dose_mean": float(pvt[dose].mean()),
                               "dose_minus_no_antibiotic": effect, "p_exact_paired": p})
            for r, q in zip(family, holm([r["p_exact_paired"] for r in family])):
                r["p_holm_across_nonzero_doses"] = q
            enrichment_rows += family
    landscape = pd.concat(landscape_rows, ignore_index=True)

    history_rows = []
    for metric in ["high_MIC_share_excl_focal", "community_weighted_MIC_score_excl_focal"]:
        family = []
        for dose in [0.0] + NONZERO_DOSES:
            x = landscape.loc[(landscape.community == "ANC_SIXTEEN") & (landscape.streptomycin == dose), metric].to_numpy(float)
            y = landscape.loc[(landscape.community == "EVO_SIXTEEN") & (landscape.streptomycin == dose), metric].to_numpy(float)
            diff, p = exact_mean_difference_test(x, y)
            family.append({"metric": metric, "streptomycin": dose, "ancestral_mean": x.mean(),
                           "evolved_mean": y.mean(), "evolved_minus_ancestral": diff, "p_exact": p})
        for r, q in zip(family, holm([r["p_exact"] for r in family])):
            r["p_holm_across_doses"] = q
        history_rows += family
    return landscape, pd.DataFrame(enrichment_rows), pd.DataFrame(history_rows)

## Plotting helpers

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import Patch



def panel_label(ax, label, dx=-36, dy=11):
    """Bold panel letter at a fixed offset (points) from the top-left corner of the axes."""
    ax.annotate(str(label).upper(), xy=(0.0, 1.0), xycoords="axes fraction", xytext=(dx, dy),
                textcoords="offset points", fontsize=10, fontweight="bold", va="bottom", ha="left",
                annotation_clip=False)


def format_grid_axis(ax, show_x=True, show_y=True, xlabel=STREP_AXIS_LABEL, ylabel=CARBON_AXIS_LABEL, ytick_every=1):
    """Axes of a carbon (rows) x streptomycin (columns) landscape."""
    ax.set_xticks(np.arange(N_STREP)); ax.set_xticklabels([f"{x:g}" for x in STREPTOMYCIN_VALUES])
    ax.set_yticks(np.arange(N_CARBON))
    ylabels = [f"{x:.2f}" for x in CARBON_VALUES]
    if ytick_every > 1:
        ylabels = [lab if k % ytick_every == 0 else "" for k, lab in enumerate(ylabels)]
    ax.set_yticklabels(ylabels)
    ax.tick_params(axis="x", labelbottom=show_x, length=2)
    ax.tick_params(axis="y", labelleft=show_y, length=2)
    ax.set_ylim(N_CARBON - 0.5, -0.5)
    if show_x:
        ax.set_xlabel(xlabel)
    if show_y:
        ax.set_ylabel(ylabel)


def draw_boundaries(ax, labels, linewidth=0.9):
    """White state boundaries with a thin dark outline."""
    labels = np.asarray(labels)
    segments = []
    for g in range(labels.shape[0]):
        for s in range(labels.shape[1] - 1):
            if labels[g, s] != labels[g, s + 1]:
                segments.append([(s + 0.5, g - 0.5), (s + 0.5, g + 0.5)])
    for g in range(labels.shape[0] - 1):
        for s in range(labels.shape[1]):
            if labels[g, s] != labels[g + 1, s]:
                segments.append([(s - 0.5, g + 0.5), (s + 0.5, g + 0.5)])
    if segments:
        lc = LineCollection(segments, colors="white", linewidths=linewidth, zorder=10,
                            capstyle="round", joinstyle="round")
        lc.set_path_effects([pe.Stroke(linewidth=linewidth + 0.95, foreground="0.15"), pe.Normal()])
        ax.add_collection(lc)


def species_plot_label(species):
    binomial = "\\ ".join(SPECIES_NAMES[species].split()[:2])
    return rf"$\it{{{binomial}}}$"


def strain_plot_label(name):
    if "\n" in name:
        binomial, qualifier = name.split("\n", 1)
        return rf"$\it{{{'\\ '.join(binomial.split())}}}$" + "\n" + qualifier
    return rf"$\it{{{'\\ '.join(name.split())}}}$"


def species_legend_handles(species_list):
    return [Patch(facecolor=SPECIES_COLORS[sp], edgecolor="none", label=species_plot_label(sp)) for sp in species_list]


def history_legend_handles():
    return [Line2D([0], [0], marker="o", color=ANCESTRAL_COLOR, lw=1.3, label=r"Ancestral $\it{A.\ caviae}$", markersize=4),
            Line2D([0], [0], marker="o", color=EVOLVED_COLOR, lw=1.3, label=r"Resistance-evolved $\it{A.\ caviae}$", markersize=4)]


def plot_segmentation_map(ax, result, show_x=True, show_y=True, descriptor=None):
    """States coloured by their dominant species, with state boundaries."""
    dom = dominant_species_map(result)
    rgb = np.ones(dom.shape + (3,), float)
    for idx, sp in enumerate(SPECIES):
        rgb[dom == idx] = to_rgb(SPECIES_COLORS[sp])
    ax.imshow(rgb, origin="upper", interpolation="nearest", aspect="auto")
    draw_boundaries(ax, result["labels_grid"], linewidth=0.9)
    format_grid_axis(ax, show_x=show_x, show_y=show_y)
    if descriptor:
        ax.text(0.5, 1.025, descriptor, transform=ax.transAxes, ha="center", va="bottom", fontsize=5.7)


def plot_composition_grid(ax, grid, labels=None, show_x=True, show_y=True, descriptor=None, ytick_every=1):
    """Stacked relative abundances in every cell of a landscape, optionally with state boundaries."""
    comp = np.clip(np.asarray(grid, float), 0, None)
    totals = comp.sum(axis=-1, keepdims=True)
    comp = np.divide(comp, totals, out=np.zeros_like(comp), where=totals > 0)
    ng, ns, _ = comp.shape
    ry, rx = 32, 12
    img = np.ones((ng * ry, ns * rx, 3), dtype=float)
    for g in range(ng):
        for st in range(ns):
            cum = np.r_[0.0, np.cumsum(comp[g, st])]; cum[-1] = 1.0
            for k, sp in enumerate(SPECIES):
                y0 = g * ry + int(np.floor(cum[k] * ry)); y1 = g * ry + int(np.ceil(cum[k + 1] * ry))
                if y1 > y0:
                    img[y0:y1, st * rx:(st + 1) * rx, :] = to_rgb(SPECIES_COLORS[sp])
    ax.imshow(img, origin="upper", interpolation="nearest", aspect="auto",
              extent=(-0.5, ns - 0.5, ng - 0.5, -0.5), rasterized=True)
    for yy in np.arange(0.5, ng - 0.5, 1):
        ax.axhline(yy, color="white", lw=0.16, alpha=0.60, zorder=3)
    for xx in np.arange(0.5, ns - 0.5, 1):
        ax.axvline(xx, color="white", lw=0.16, alpha=0.60, zorder=3)
    if labels is not None:
        draw_boundaries(ax, labels, linewidth=0.95)
    format_grid_axis(ax, show_x=show_x, show_y=show_y, ytick_every=ytick_every)
    if descriptor:
        ax.text(0.5, 1.025, descriptor, transform=ax.transAxes, ha="center", va="bottom", fontsize=5.7)


def replicate_mean_ci(ax, x0, vals, color, seed=1, jitter_width=0.055, size=13):
    """Replicates as open points and their mean with a 95% confidence interval."""
    vals = np.asarray(vals, float)
    vals = vals[np.isfinite(vals)]
    rng = np.random.default_rng(seed)
    jitter = rng.uniform(-jitter_width, jitter_width, len(vals)) if len(vals) else np.array([])
    ax.scatter(x0 + jitter, vals, s=size, facecolor="white", edgecolor=color, linewidth=0.75, zorder=3)
    if len(vals):
        sem = vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0
        ax.errorbar(x0, vals.mean(), yerr=1.96 * sem, fmt="o", mfc=color, mec=color, color=color,
                    ms=4.4, lw=1.1, capsize=2, zorder=4)


def save_figure(fig, name):
    """Save a figure as PDF and 300-dpi PNG in results/figures."""
    path = FIGURE_DIR / f"{name}.pdf"
    fig.savefig(path, format="pdf", metadata={"CreationDate": None})  # reproducible files
    fig.savefig(path.with_suffix(".png"), format="png", dpi=300)
    plt.close(fig)